<a href="https://colab.research.google.com/github/arishAI-08/perceptron-ev-adoption/blob/main/perceptron_ev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from google.colab import files

np.random.seed(42)

uploaded = files.upload()

Saving EV_Adoption_and_Range_Anxiety_Dataset.csv to EV_Adoption_and_Range_Anxiety_Dataset.csv


In [3]:
# 2. Load the Dataset

df = pd.read_csv("EV_Adoption_and_Range_Anxiety_Dataset.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 15)


,Buyer_ID,Age,Gender,Annual_Income_USD,City_Type,Daily_Commute_km,Number_of_Cars_Owned,Current_Car_Type,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Home_Charging_Possible,Environmental_Concern_Level,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,EV00001,56,Male,44906.0,Rural,18.1,1,Truck,1,0,Yes,4.0,Yes,Low,No
1,EV00002,36,Male,88754.0,Rural,5.0,1,SUV,2,3,Yes,1.0,Yes,Low,No
2,EV00003,42,Male,155958.0,Urban,32.2,2,Sedan,14,3,No,2.0,Yes,Medium,Yes
3,EV00004,31,Male,37059.0,Rural,89.9,1,Sedan,2,1,Yes,4.0,No,Low,No
4,EV00005,48,Female,48552.0,Rural,71.0,1,Sedan,2,0,Yes,3.0,Yes,Low,No


In [4]:
# 3. preprocessing
df = df.drop(columns=["Buyer_ID"])  # ID is not a feature

# Fill missing values with the column median
for col in ["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]:
    df[col] = df[col].fillna(df[col].median())

# Convert text columns to numbers
df["Home_Charging_Possible"] = df["Home_Charging_Possible"].map({"No": 0, "Yes": 1})
df["Subsidy_Available"]      = df["Subsidy_Available"].map({"No": 0, "Yes": 1})
df["Range_Anxiety_Level"]    = df["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2})
df["Will_Buy_EV"]            = df["Will_Buy_EV"].map({"No": 0, "Yes": 1})  # target

# One-hot encode the remaining categorical columns
df = pd.get_dummies(df, columns=["Gender", "City_Type", "Current_Car_Type"], dtype=int)

# Features (X) and target (y)
y = df["Will_Buy_EV"].values
X = df.drop(columns=["Will_Buy_EV"]).values.astype(float)

print("Missing values left:", df.isna().sum().sum())
print("X shape:", X.shape)

Missing values left: 0
X shape: (10000, 20)


In [5]:
# 4. 80/20 split
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]

# Standardize using TRAIN statistics only
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
std[std == 0] = 1
X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (8000, 20) | Test: (2000, 20)


In [6]:
# 5. Perceptron class

class Perceptron:
    def __init__(self, n_features, lr=0.01, epochs=50):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.lr = lr
        self.epochs = epochs

    def predict(self, X):
        return (np.dot(X, self.w) + self.b >= 0).astype(int)  # step function

    def fit(self, X, y):
        for epoch in range(self.epochs):
            errors = 0
            for xi, target in zip(X, y):
                error = target - self.predict(xi.reshape(1, -1))[0]
                # Perceptron learning rule
                self.w += self.lr * error * xi
                self.b += self.lr * error
                errors += int(error != 0)
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1:3d} | misclassified: {errors}")

In [7]:
# 6. model training

model = Perceptron(n_features=X_train.shape[1], lr=0.01, epochs=50)
model.fit(X_train, y_train)

Epoch  10 | misclassified: 1423
Epoch  20 | misclassified: 1452
Epoch  30 | misclassified: 1445
Epoch  40 | misclassified: 1436
Epoch  50 | misclassified: 1414


In [8]:
# 7. evaluation
y_pred = model.predict(X_test)

TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

accuracy  = (TP + TN) / len(y_test)
precision = TP / (TP + FP) if (TP + FP) else 0
recall    = TP / (TP + FN) if (TP + FN) else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print("---- FINAL RESULTS ----")
print("Confusion Matrix:")
print("               Pred No  Pred Yes")
print(f"Actual No     {TN:7d}  {FP:8d}")
print(f"Actual Yes    {FN:7d}  {TP:8d}")
print(f"\nAccuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

---- FINAL RESULTS ----
Confusion Matrix:
               Pred No  Pred Yes
Actual No        1498       167
Actual Yes        182       153

Accuracy : 0.8255
Precision: 0.4781
Recall   : 0.4567
F1-Score : 0.4672
